<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z317_AutoGluon_Galactus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AutoGluon  -  Empiojado modo GALACTUS

**Idea**: igual que el empiojado Galactus del AutoARIMA, pero aplicado a AutoGluon.

Para cada `product_id` se **desordena** (shuffle) el vector de `tn` a lo largo de los 36 meses, manteniendo intacto el eje `timestamp`.

Se mantiene:
* el promedio de los 36 meses por producto

Se destruye:
* el promedio de los ultimos 12 meses
* el valor del año anterior (mismo mes)
* la tendencia
* la estacionalidad

Si el score de Kaggle del shuffleado **no es mucho peor** que el del dataset real, AutoGluon no esta aprovechando la estructura temporal. Si cae fuerte, hay señal real.

*Natalia Galactus, la destructora de mundos*

## 0.1 Init ambiente Google Colab

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

# 1  Modelo AutoGluon - Galactus

## 1.1 Init Experimento

In [ ]:
# instalacion de paquetes que NO vienen por default en Colab
# autogluon es MUY pesado. varios minutos se perderan aqui
!pip install uv
!uv pip install -q kaggle
!uv pip install autogluon[all]

In [ ]:
# funcion para hacer submits a Kaggle
def kaggle_submit(competencia, archivo, mensaje):

  # comando
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  # ejecucion
  os.system(comando)

In [ ]:
import os as os
import numpy as np
import polars as pl

from datetime import datetime
from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

Por favor, cargar aqui SU semilla primigenia
<br> **Muy importante**, cambiar el numero de experimento en cada corrida. Usted ha sido notificado !
<br> Si cada corrida no está en una nueva carpeta virgen, entonces se reutilizaran modelos viejos corridos con otros parametros

In [ ]:
# defino los parametros
PARAM = {'experimento':'AutoGluon_Galactus01',
  'kaggle_competition':'labo-iii-2026-rosario',
  'semilla_primigenia':102191
}

In [ ]:
# creo la carpeta del experimento y hago el chdir
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

## 1.2 Init AutoGluon

In [ ]:
# cargo el dataset del sell-in
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

In [ ]:
# agrupo por product_id, periodo
tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
)

tb_ventas = tb_ventas.sort(["product_id", "periodo"])

In [ ]:
# cargo la tabla "apredecir" que contiene los 780 productos que deben predecirse las ventas de 202002
tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")


# Filtro tb_ventas a solo las que debo predecir
print(tb_ventas.height)
tb_ventas = tb_ventas.join(tb_apredecir,
  on="product_id",
  how="inner"
)
print(tb_ventas.height)
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

In [ ]:
# paso de periodo a  timestamp
tb_ventas = tb_ventas.with_columns(
    (pl.col('periodo').cast(pl.String).str.to_datetime('%Y%m')).alias('timestamp')
)

## 1.2.bis  Empiojado GALACTUS  -  shuffle de cada serie

Para cada `product_id` desordenamos al azar los valores de `tn` a lo largo de los 36 meses, manteniendo intacto el eje `timestamp`.

Resultado:
* media global de la serie: **intacta**
* promedio de los ultimos 12 meses: **destruido**
* valor del año anterior (mismo mes): **destruido**
* tendencia: **destruida**
* estacionalidad: **destruida**

Si AutoGluon es bueno aprovechando estructura temporal, su score en Kaggle deberia caer fuerte respecto del dataset real. Si cae poco, indica que las series son intrinsecamente ruidosas o que AutoGluon esta cerca de un promedio.

In [ ]:
empiojar_galactus = True

if empiojar_galactus:
  np.random.seed(PARAM['semilla_primigenia'])

  tb_ventas = tb_ventas.sort(["product_id", "periodo"])

  # shuffleo los valores de tn dentro de cada product_id
  # el eje timestamp queda intacto - solo se desordenan los valores
  piezas = []
  for pid in tb_ventas["product_id"].unique(maintain_order=True).to_list():
    sub = tb_ventas.filter(pl.col("product_id") == pid)
    tn_vals = sub["tn"].to_numpy().copy()
    np.random.shuffle(tn_vals)
    piezas.append(sub.with_columns(pl.Series("tn", tn_vals)))

  tb_ventas = pl.concat(piezas).sort(["product_id", "periodo"])

  print("empiojado GALACTUS aplicado - serie shuffleada por producto")

In [ ]:
# verificacion: la media por producto se conserva
display(
  tb_ventas.group_by("product_id").agg(pl.col("tn").mean().alias("media_36m")).sort("product_id").head(10)
)

## 1.3 Entrenamiento AutoGluon

Mismo setup que `z316_AutoGluon.ipynb`. El unico cambio es el shuffle previo de la serie.

Como los timestamps son los mismos pero los valores estan shuffleados, AutoGluon recibe series sin estructura temporal real.

In [ ]:
# paso a formato  ts = time_series
ts_data = TimeSeriesDataFrame.from_data_frame(
  tb_ventas.to_pandas(), # tristemente AutoGluon espera un dataframe Pandas ...
  timestamp_column='timestamp',
  id_column='product_id'
)

In [ ]:
# Entrenamiento, esta es la parte pesada
global_eval_metric = 'RMSE'

# defino
modelo = TimeSeriesPredictor(
  prediction_length= 2,  # horizonte de prediccion
  target= 'tn',
  freq= 'MS',  # Frecuencia mensual (Month Start)
  eval_metric= global_eval_metric
)

# entreno con los mismos hiperparametros que la corrida real
# el unico cambio es que la serie esta shuffleada
modelo.fit(ts_data,
  num_val_windows= 2,
  time_limit= 3600, # una hora
  presets= "best_quality",
  random_seed= PARAM['semilla_primigenia']
)

## 1.4 Prediccion AutoGluon

In [ ]:
# predict a partir los mismos datos

tb_forecast = modelo.predict(ts_data,
  random_seed= PARAM['semilla_primigenia']
)

display(tb_forecast)

In [ ]:
# paso a formato Polars, teniendo en cuenta el indice
tb_forecast = pl.from_pandas(tb_forecast.reset_index())

In [ ]:
# me quedo con las predicciones de febrero-2020
# en 'mean' esta la prediccion, mas alla de los n-tiles
tb_final = tb_forecast.filter(pl.col("timestamp") ==  datetime(2020, 2, 1)).select(["item_id","mean"])

display(tb_final)

## 1.5 Submit a Kaggle

In [ ]:
# cambio nombre de campos a los que reconoce Kaggle
tb_final = tb_final.rename({
  "item_id": "product_id",
  "mean": "tn",
})

In [ ]:
# Submit a Kaggle
archivo= "AutoGluon_Galactus_" + global_eval_metric + ".csv"
mensaje= "AutoGluon empiojado GALACTUS shuffle de la serie " + global_eval_metric

tb_final.write_csv(archivo)

kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje )

## 1.6  Comparacion con la corrida real

Una vez que tengas los dos scores en Kaggle (la corrida normal `z316_AutoGluon` y esta corrida Galactus), compara:

| Resultado | Interpretacion |
| --- | --- |
| Galactus score ≈ score real | AutoGluon esta cerca de un promedio: la estructura temporal no aporta mucho, o el algoritmo no la esta aprovechando. Series intrinsecamente ruidosas. |
| Galactus score >> score real (mucho peor) | AutoGluon SI aprovecha estacionalidad / tendencia / correlacion entre series. Vale la pena seguir invirtiendo en modelos temporales. |
| Galactus score < score real | Caso raro: la estructura temporal estaba confundiendo al modelo. Probablemente convenga suavizar la serie antes de entrenar. |